# 01 Data Collection

## Literature Review

The 15-minute city has become one of the most influential recent planning ideas because it shifts
attention away from mobility as speed alone and toward mobility as everyday access. Moreno et al.
(2021) frame the model as a human-centred urbanism in which essential activities should be reachable
within a short walk or bike ride. Their argument is not only environmental. It also links proximity
to resilience, public health, local identity, and the reduction of forced car dependence. For a city
like Shanghai, this matters because a metropolitan system can perform well at a macro scale while still
producing highly uneven neighbourhood-level access to daily services. In that sense, the 15-minute city
is best understood as an accessibility project rather than a simple transport project.

That distinction is important in light of the earlier accessibility literature. Geurs and van Wee
(2004) argue that accessibility should be evaluated through multiple interacting components: land use,
transport, time, and the characteristics of individuals. Their review remains useful because it warns
against reducing accessibility to distance or travel time alone. A neighbourhood may be close to many
destinations, but the quality, affordability, or suitability of those destinations also matters.
Likewise, the same spatial structure can be experienced very differently by walkers, cyclists, transit
users, older adults, or lower-income households. This project follows that logic by combining service
density, transport structure, housing cost proxies, and track-specific amenities instead of relying on
a single nearest-destination measure.

The equity dimension is equally central. Lucas (2012) shows that transport disadvantage and social
exclusion are deeply connected, particularly when essential opportunities are spatially available only
to households that can afford the time, money, or mode required to reach them. This critique matters
for 15-minute city work because proximity can easily become a premium urban good. If high-access areas
are systematically more expensive, then a city may score well on livability while still excluding many
residents from those advantages. For Shanghai, where land values and neighbourhood status vary sharply
across the inner city, new towns, and peripheral districts, a 15-minute analysis should therefore ask
not only where amenities cluster, but also who can realistically benefit from them.

Pozoukidou and Chatziyiannaki (2021) extend this discussion by treating the 15-minute city as both a
planning model and a political imagination. They argue that proximity-based planning cannot be reduced
to a decorative slogan about compactness; it must be operationalised through measurable indicators of
walkability, mixed functions, and neighbourhood regeneration. At the same time, they warn that the
model can become utopian if planners ignore institutional capacity, spatial inequality, and the uneven
quality of the urban fabric. This is a useful warning for a Shanghai workflow because it suggests that
an analytical pipeline should stay explicit about its assumptions, scales, and blind spots. A map of
"good" 15-minute areas is only meaningful if the scoring logic is transparent and the missing data are
clearly documented.

The prototype developed in this project therefore adopts three principles from the literature. First,
it treats accessibility as a multi-dimensional condition rather than a single travel metric. Second, it
keeps equity visible by pairing amenity access with a housing-cost proxy. Third, it builds the analysis
so that methods can be upgraded in stages: starting from cell-level and H3 proxy indicators, then moving
toward fully network-based isochrones and richer real-time data once the pipeline is stable. This staged
strategy is especially appropriate for a five-week intensive project, where reproducibility and design
clarity matter as much as model sophistication.

In practical terms, the literature supports a workflow that begins with raw spatial datasets, validates
their provenance, constructs a common spatial framework, and only then derives scores. That order is not
administrative busywork. It is what makes later interpretation credible. If the 15-minute city is about
everyday life, then the analysis must remain anchored in documented sources, defensible category choices,
and readable visual outputs. The notebooks in this prototype are structured around that logic: notebook
01 documents sources and cleaning, notebook 02 defines the grid and accessibility logic, and notebook 03
makes the scoring and H3 aggregation fully inspectable. The result is not yet a finished urban policy
instrument, but it is a reproducible analytical foundation for one.

### References

1. Moreno, C., Allam, Z., Chabaud, D., Gall, C., & Pratlong, F. (2021). *Introducing the “15-Minute City”:
   Sustainability, Resilience and Place Identity in Future Post-Pandemic Cities*. Smart Cities, 4(1), 93-111.
2. Geurs, K. T., & van Wee, B. (2004). *Accessibility evaluation of land-use and transport strategies:
   review and research directions*. Journal of Transport Geography, 12(2), 127-140.
3. Lucas, K. (2012). *Transport and social exclusion: Where are we now?* Transport Policy, 20, 105-113.
4. Pozoukidou, G., & Chatziyiannaki, Z. (2021). *15-Minute City: Decomposing the New Urban Planning Eutopia*.
   Sustainability, 13(2), 928.


## Project Data Inventory

This notebook documents the raw and processed inputs currently used by the prototype:

- `UTSEUS-anjuke-real-estate.csv`
- `POI 2024.zip`
- extracted classified 2024 POI CSVs under `data/raw/poi_2024/POI 2024/csv格式/已分类`
- `shanghai-roads-simplified.parquet`
- Shanghai municipal boundary from Aliyun DataV

The current prototype uses the 2024 classified POI extracts as its main amenity layer.
Type-based filtering notes are stored in `poi_2024_probe.json` and `poi_2024_mapping_notes.md`.
It also produces a `500 m` grid-level processed layer before aggregation to H3.


## Source Provenance Log

The project brief requires at least four distinct datasets and explicit provenance logging. The current
prototype uses five source groups: housing, POI amenities, road network, municipal boundary, and
environmental proxies. The manifest below stores the public-facing provenance log without exposing
machine-specific absolute paths.


In [ ]:
from pathlib import Path
import json
import os
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = Path(os.environ.get("SHANGHAI_15MC_RAW_DIR", ROOT / "data" / "raw"))
APARTMENT_PATH = Path(os.environ.get("SHANGHAI_15MC_APARTMENT_PATH", ROOT / "data" / "raw" / "anjuke_price_distance_filtered.parquet"))
if not APARTMENT_PATH.exists():
    APARTMENT_PATH = ROOT.parent / "anjuke_price_distance_filtered.parquet"
ROADS_PATH = Path(os.environ.get("SHANGHAI_15MC_ROADS_PATH", ROOT / "data" / "raw" / "shanghai-roads-simplified.parquet"))
PROCESSED = ROOT / "data" / "processed"

inventory = [
    RAW_DIR / "UTSEUS-anjuke-real-estate.csv",
    RAW_DIR / "POI 2024.zip",
    APARTMENT_PATH,
    ROADS_PATH,
    ROOT / "data" / "raw" / "poi_2024" / "POI 2024" / "csv格式" / "已分类",
    PROCESSED / "project_manifest.json",
    PROCESSED / "poi_2024_probe.json",
    PROCESSED / "shanghai_grid_seed.json",
]

pd.DataFrame(
    [
        {"file": str(p), "exists": p.exists(), "size_mb": round(p.stat().st_size / 1_048_576, 2) if p.exists() else None}
        for p in inventory
    ]
)

## Processed Prototype Manifest

In [ ]:
manifest = json.loads((PROCESSED / "project_manifest.json").read_text(encoding="utf-8"))
manifest

In [ ]:
pd.DataFrame(manifest["source_provenance"])[[
    "id",
    "source",
    "type",
    "role",
    "collection_date",
    "processing_note",
]]

## Apartment Data Preview

In [ ]:
apartment = pd.read_parquet(APARTMENT_PATH)
apartment[["longitude", "latitude", "onesquaremeter", "distances"]].describe().T

## Road Network Preview

In [ ]:
roads = pd.read_parquet(ROADS_PATH)
roads.head()

## AI Assistance And Integrity Note

AI assistance was used for code scaffolding, debugging, documentation drafting, local QA, and app
iteration. The final responsibility for source compliance, interpretation, weighting choices, and
submission decisions remains with the student. See `AI_ASSISTANCE.md` in the project root.


## Final Source-Review Notes

The current local prototype already logs source provenance in `project_manifest.json`. Before public
submission, review source terms and append any final API-based collection details, especially if the
proxy workflow is upgraded with routing or GTFS data:

1. how the 2024 POI archive was decoded and refreshed
2. any API usage for routing or GTFS retrieval
3. data collection dates for each layer
4. licensing / terms-of-service checks
5. a reproducible source log for all derived files

The static project files do not include API keys or private credentials.
